# Jupyter Notebook: Decision Trees

## Step 1: Install Required Libraries

First, ensure you have the necessary libraries installed. You can install them using the following command:

`pip install pandas numpy seaborn matplotlib scikit-learn`

In [ ]:
import matplotlib.pyplot as plt
# Imports
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

## Step 2: Reading the data and data cleaning
Reading the training data

In [ ]:
df = pd.read_csv(r'train.csv')  # Replace by your path
df.head(5)

Determining how many missing values each column contains. 

In [ ]:
column_names = df.columns
for column in column_names:
    missing_count = df[column].isnull().sum()
    total_count = len(df[column])
    missing_percentage = (missing_count / total_count) * 100
    print(f'{column} - {missing_count}, {missing_percentage:.2f}%')

* Age has ~20% missing values. This will require handling.
* Cabin has ~77% missing values and might be excluded from analysis.
* Embarked has a small proportion of missing values. This will require handling.

Filling missing values for 'age' with the median. The median is used here instead of the mean because it is less affected by outliers. The 'age' column might have extreme values (very young or very old passengers), and the median provides a more robust measure.

In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].median())

Filling missing values for 'Embarked' with the mode.
The 'Embarked' column contains categorical data. Using the mode (most frequent value) ensures that the imputed value makes sense contextually.

In [ ]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

Dropping 'Cabin' due to a large number of missing values
The 'Cabin' column has too many missing values to be useful for analysis. Dropping it simplifies the dataset.

In [ ]:
df = df.drop(columns=['Cabin'])

Let's make sure that there are no more missing values. 

In [ ]:
column_names = df.columns
for column in column_names:
    missing_count = df[column].isnull().sum()
    total_count = len(df[column])
    missing_percentage = (missing_count / total_count) * 100
    print(f'{column} - {missing_count}, {missing_percentage:.2f}%')

In [ ]:
# Dropping the passengerId, Name, and Ticket
df = df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)

In [ ]:
# Convert categorical to numeric
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

## Step 3: Separating the target value (Survived) from the rest 
Separating the features from the target

In [ ]:
# Features and target
X = df.drop('Survived', axis=1)
y = df['Survived']

## Step 4: Spliting the data 
Dividing the data into train and validation sets (Kaggle has already separated a test set) 

In [ ]:
# Train-val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 5: Training the decision tree classifier
Defining the model and training the model. 

In [ ]:
# Create and train the model
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# Step 6: Calculate the performance of the model
We are going to use some numerical metrics to calculate the performance of the algorithm

In [ ]:

# Predictions
y_pred = model.predict(X_val)

# Evaluation
print("Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))

# Step 7: Visualize the decision tree classifier

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(model, filled=True, feature_names=X.columns, class_names=['Not Survived', 'Survived'])
plt.title("Decision Tree")
plt.show()

## Investigation tasks

**Task 1**: Explain how you can interpret a confusion matrix and interpret the confusion matrix obtained in step 6.

<span style="color:red">(1 mark)</span>

               precision    recall  f1-score   support

           0       0.82      0.80      0.81       105
           1       0.73      0.76      0.74        74

    accuracy                           0.78       179
   macro avg       0.78      0.78      0.78       179
weighted avg       0.78      0.78      0.78       179


Confusion Matrix:
 [[84 21]
 [18 56]]

The confusion matrix is can be interpreted via the first column/row (1,1) being the amount of positives being predicted as positive, the (1,2) spot means the amount of Negative being positive, (2,1) is the amount of negative being positive, (2,2) is the amount of negative being predicted negative.

**Task 2**: In machine learning, various metrics are used to evaluate the performance of classification algorithms. Two of the most commonly used are Accuracy and F1-Score.
Research and explain what each of these metrics represents, including their mathematical formulas. Then, discuss which metric is more appropriate when dealing with an imbalanced dataset (e.g., when the number of survivors and non-survivors is not equal, as in the Titanic dataset). Provide a clear justification for your choice.

<span style="color:red">(2 marks)</span>

Accuracy is the proportion of correct prediction out of all predictions, it is below:
$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$
F1 Score is a metric that combines precision and recall score, scored between 0 and 1, being calculated from the below formula:
$$
F_1 = \frac{2 \cdot (\text{precision} \cdot \text{recall})}{\text{precision} + \text{recall}}
$$
In this case, the most correct data to pick is F1 score, as a model with high accuracy could be returning undesired predictions in a imbalanced data set, like always returning False.

**Task 3**: Calculate the Accuracy and F1-Score for both the training and validation datasets. Then, analyse and interpret the results, what do the metrics suggest about your model’s performance? Is there any evidence of overfitting or underfitting?

<span style="color:red">(2 marks)</span>

The model accuracy is 0.78, with it being good but not great, also alluding to it performing better on the majority class than the minority class. The f1 score is 0.81 on those whom did not survive, which is pretty good however it is 0.74 on those whom survived, therefore impling that there is evidence of overfitting.

**Task 4**: Train a new decision tree using only 'Sex_male', and 'Age' as features. Calculate and interpret the performance on training and validation. Compare the results to the previous model.

<span style="color:red">(2 marks)</span>

In [ ]:
X = df[['Sex_male', 'Age']]
y = df['Survived']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)


**Task 5**: Visualize new decision tree and interpret it. 

<span style="color:red">(1 mark)</span>

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(model, filled=True, max_depth=2, feature_names=X.columns, class_names=['Not Survived', 'Survived'])
plt.title("Decision Tree")
plt.show()

The decision tree works via classifying and splitting the data set up, the first question is the most important question, as seen above the question was in regards to its gender, asking whether it is male or female.

**Task 6**: When training a Decision Tree Classifier, several parameters can be adjusted to control the model’s complexity and improve its performance, particularly in addressing issues like overfitting and underfitting.

Using the [scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html), choose three parameters of the DecisionTreeClassifier. For each parameter:

- Describe what it does

- Explain how changing its value could influence overfitting or underfitting in the model

<span style="color:red">(3 marks)</span>

1. Max Depth: What it does is that it limits the maximum depth of the tree. Changing this parameter will affect how big the tree can grow, if it is too big it can lead to overfitting, if it is too small it can lead to underfitting.
2. Min Samples Split: It prevents splitting on small sample sizes, controlling the minimum number of samples required to split an internal node. If the value is too low, it can result in agressive splitting -> Overfit. If the value is too high -> Underfit.
3. Min Samples Leaf: It controls the minimum number of samples required to be at a leaf node... ensuring such split leaaves will at least have taht many samples in both branches. In common terms, it means that it rejects the splitting of a parent node which whould leave to its children having less than the specified min number. Preventing overfitting with a small value, but with a larger value can result in underfitting.

**Task 7**: Select a few different values for the three parameters you previously explored and train multiple Decision Tree models using these combinations. 

- Evaluate each model using a performance metric of your choice (e.g., Accuracy or F1-Score) on both the training and validation sets

- Compare the results across the different parameter settings

- Identify which combination of parameters provides the best overall performance


<span style="color:red">(4 marks)</span>


### First trying to optimise for Max Depth

In [ ]:
for max_depth in [3, 5, 10, None]:
    model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    print(f"Max Depth: {max_depth}")
    print("Training Accuracy:", accuracy_score(y_train, y_pred_train))
    print("Validation Accuracy:", accuracy_score(y_val, y_pred_val))
    print("f1 Score (Training):", classification_report(y_train, y_pred_train))
    print("f1 Score (Validation):", classification_report(y_val, y_pred_val))
    print("\n")

Depth 3 + 5 = 0.782\
Depth 10 = 0. 776 -> Worse\
Training accuracy increases with more depth, overfitting increases as depth increases\
Best model is depth 3


### Trying to optimise for Min Samples Split

In [ ]:
for min_samples_split in [2, 5, 10]:
    model = DecisionTreeClassifier(min_samples_split=min_samples_split, random_state=42)
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    print(f"Min Samples Split: {min_samples_split}")
    print("Training Accuracy:", accuracy_score(y_train, y_pred_train))
    print("Validation Accuracy:", accuracy_score(y_val, y_pred_val))
    print("f1 Score (Training):", classification_report(y_train, y_pred_train))
    print("f1 Score (Validation):", classification_report(y_val, y_pred_val))
    print("\n")

There is no improvement from 2 -> 5 with 10 + being worse\
This parameter has almost no effect\
Best case is to use < 10

### Trying to optimise for Min Samples Leaf

In [ ]:
for min_samples_leaf in [1, 2, 5]:
    model = DecisionTreeClassifier(min_samples_leaf=min_samples_leaf, random_state=42)
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_val = model.predict(X_val)
    print(f"Min Samples Leaf: {min_samples_leaf}")
    print("Training Accuracy:", accuracy_score(y_train, y_pred_train))
    print("Validation Accuracy:", accuracy_score(y_val, y_pred_val))
    print("f1 Score (Training):", classification_report(y_train, y_pred_train))
    print("f1 Score (Validation):", classification_report(y_val, y_pred_val))
    print("\n")

Increasing min_samples_leaf hurts performance\
Best case is to use 1


### Summarising the best parameters
The best parameters are max_depth = 3, min_samples_split < 10, min_samples_leaf = 1, with the paramaters that having the highest effect being max_depth, with the other two having a very small effect on the model performance.

**Bonus**: Use GridSearch and RandomSearch to find the best combination of parameters. 

<span style="color:red">(0.5 marks)</span>

### I don't pay electricity bills so lets use grid search

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': [2, 3, 4, 5, 7, 10],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 5],
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_features': ['sqrt', 'log2', None],
    'class_weight': [None, 'balanced'],
    'min_weight_fraction_leaf': [0.0, 0.05, 0.1]
}

print("In progress...")

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
y_pred_grid = best_model.predict(X_val)

print(f"Best Params: {best_params}")
print(f"Best CV F1 (weighted): {grid_search.best_score_:.4f}")
print(f"Validation Accuracy: {accuracy_score(y_val, y_pred_grid):.4f}")
print(classification_report(y_val, y_pred_grid))

**Bonus**: Try to improve the results as much as possible. Some ideas of things you could try:

- Testing which are the best features to train the algorithm
- Cross-validation
- Majority voting
- Other classification algorithms
- A combination of classification algorithms

<span style="color:red">(Up to 2 marks)</span>

## Plan is to use gradient boosted trees
We'll optimize using weighted F1 only: test feature sets, tune multiple classifiers, and compare against majority voting.

In [46]:
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score

# Rebuild full feature set (Task 4 narrowed X to two columns)
X_full = df.drop('Survived', axis=1)
y_full = df['Survived']

X_train_full, X_val_full, y_train_full, y_val_full = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1) Feature-set trials
manual_features = [
    col for col in ['Pclass', 'Sex_male', 'Age', 'Fare', 'SibSp', 'Parch', 'Embarked_Q', 'Embarked_S']
    if col in X_full.columns
]

selector = SelectFromModel(
    ExtraTreesClassifier(n_estimators=1200, random_state=42, n_jobs=-1),
    threshold='median'
)
selector.fit(X_train_full, y_train_full)
importance_features = X_train_full.columns[selector.get_support()].tolist()

feature_sets = {
    'all_features': X_full.columns.tolist(),
    'manual_core_features': manual_features,
    'importance_selected_features': importance_features,
}

feature_results = []
for feature_name, cols in feature_sets.items():
    probe_model = GradientBoostingClassifier(random_state=42)
    feature_score = cross_val_score(
        probe_model,
        X_train_full[cols],
        y_train_full,
        cv=cv,
        scoring='f1_weighted',
        n_jobs=-1,
    ).mean()
    feature_results.append((feature_name, feature_score, cols))

feature_results = sorted(feature_results, key=lambda x: x[1], reverse=True)
best_feature_set_name, best_feature_score, best_columns = feature_results[0]

print('Best feature set:', best_feature_set_name)
print('Feature count:', len(best_columns))
print(f'Best feature-set CV Weighted F1: {best_feature_score:.4f}')

X_train_best = X_train_full[best_columns]
X_val_best = X_val_full[best_columns]

# 2) Deep model tuning
search_space = {
    'logistic_regression': (
        LogisticRegression(max_iter=8000, random_state=42),
        {
            'C': [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100],
            'solver': ['lbfgs', 'liblinear', 'saga'],
            'class_weight': [None, 'balanced'],
        },
        60,
    ),
    'random_forest': (
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {
            'n_estimators': [300, 500, 800, 1200],
            'max_depth': [None, 4, 6, 8, 10, 14],
            'min_samples_split': [2, 4, 6, 10, 15],
            'min_samples_leaf': [1, 2, 3, 4, 6],
            'max_features': ['sqrt', 'log2', None],
            'class_weight': [None, 'balanced', 'balanced_subsample'],
        },
        120,
    ),
    'extra_trees': (
        ExtraTreesClassifier(random_state=42, n_jobs=-1),
        {
            'n_estimators': [300, 500, 800, 1200],
            'max_depth': [None, 4, 6, 8, 10, 14],
            'min_samples_split': [2, 4, 6, 10, 15],
            'min_samples_leaf': [1, 2, 3, 4, 6],
            'max_features': ['sqrt', 'log2', None],
            'class_weight': [None, 'balanced', 'balanced_subsample'],
        },
        120,
    ),
    'gradient_boosting': (
        GradientBoostingClassifier(random_state=42),
        {
            'n_estimators': [100, 200, 400, 800],
            'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.1],
            'max_depth': [2, 3, 4, 5],
            'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
            'min_samples_split': [2, 4, 6, 10],
            'min_samples_leaf': [1, 2, 3, 4],
        },
        120,
    ),
}

model_results = []
tuned_models = {}

for model_name, (estimator, param_dist, n_iter) in search_space.items():
    print(f'Searching {model_name}...')
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring='f1_weighted',
        cv=cv,
        n_jobs=-1,
        random_state=42,
        refit=True,
    )
    search.fit(X_train_best, y_train_full)

    tuned_models[model_name] = search.best_estimator_
    model_results.append(
        {
            'model': model_name,
            'cv_weighted_f1': search.best_score_,
            'best_params': search.best_params_,
        }
    )

results_df = pd.DataFrame(model_results).sort_values('cv_weighted_f1', ascending=False).reset_index(drop=True)
print(results_df[['model', 'cv_weighted_f1']])

# 3) Majority voting with top 3 tuned models
top_names = results_df['model'].head(3).tolist()
hard_voter = VotingClassifier(
    estimators=[(name, clone(tuned_models[name])) for name in top_names],
    voting='hard',
    n_jobs=-1,
)
soft_voter = VotingClassifier(
    estimators=[(name, clone(tuned_models[name])) for name in top_names],
    voting='soft',
    n_jobs=-1,
)

hard_score = cross_val_score(hard_voter, X_train_best, y_train_full, cv=cv, scoring='f1_weighted', n_jobs=-1).mean()
soft_score = cross_val_score(soft_voter, X_train_best, y_train_full, cv=cv, scoring='f1_weighted', n_jobs=-1).mean()

ensemble_scores = pd.DataFrame(
    [
        {'model': 'voting_hard_top3', 'cv_weighted_f1': hard_score},
        {'model': 'voting_soft_top3', 'cv_weighted_f1': soft_score},
    ]
).sort_values('cv_weighted_f1', ascending=False)

print(ensemble_scores)

# 4) Final winner by weighted F1 only
best_single_name = results_df.iloc[0]['model']
best_single_score = results_df.iloc[0]['cv_weighted_f1']
best_ensemble_name = ensemble_scores.iloc[0]['model']
best_ensemble_score = ensemble_scores.iloc[0]['cv_weighted_f1']

if best_ensemble_score > best_single_score:
    final_model_name = best_ensemble_name
    final_model = soft_voter if best_ensemble_name == 'voting_soft_top3' else hard_voter
    final_cv_weighted_f1 = best_ensemble_score
    final_best_params = 'Ensemble of top 3 tuned models'
else:
    final_model_name = best_single_name
    final_model = tuned_models[best_single_name]
    final_cv_weighted_f1 = best_single_score
    final_best_params = results_df.loc[0, 'best_params']

final_model.fit(X_train_best, y_train_full)
y_pred_final = final_model.predict(X_val_best)
final_report = classification_report(y_val_full, y_pred_final, output_dict=True)

print('Final Model:', final_model_name)
print('Final Best Params:', final_best_params)
print(f'Final CV Weighted F1: {final_cv_weighted_f1:.4f}')
print(f"Validation Weighted F1: {final_report['weighted avg']['f1-score']:.4f}")
print(classification_report(y_val_full, y_pred_final))


Best feature set: importance_selected_features
Feature count: 4
Best feature-set CV Weighted F1: 0.8219
Searching logistic_regression...


/Users/scott/Library/Python/3.13/lib/python/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 54 is smaller than n_iter=60. Running 54 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Searching random_forest...
Searching extra_trees...
Searching gradient_boosting...
                 model  cv_weighted_f1
0    gradient_boosting        0.838684
1        random_forest        0.837901
2          extra_trees        0.828909
3  logistic_regression        0.793576
              model  cv_weighted_f1
0  voting_hard_top3        0.839900
1  voting_soft_top3        0.835586
Final Model: voting_hard_top3
Final Best Params: Ensemble of top 3 tuned models
Final CV Weighted F1: 0.8399
Validation Weighted F1: 0.8355
              precision    recall  f1-score   support

           0       0.84      0.91      0.87       110
           1       0.83      0.72      0.78        69

    accuracy                           0.84       179
   macro avg       0.84      0.82      0.82       179
weighted avg       0.84      0.84      0.84       179



## Bonus 2 improved workflow
This version reloads the raw Titanic data, engineers richer passenger features, compares feature spaces with out-of-fold weighted F1, calibrates the chosen probability threshold, and then retrains on all labelled rows before writing a submission file.

In [ ]:
from bonus2_boosted_workflow import run_bonus2_workflow

bonus2_improved_results = run_bonus2_workflow(
    train_path='train.csv',
    test_path='test.csv',
    submission_path='bonus2_improved_submission.csv',
)
bonus2_improved_results['leaderboard'].head(8)
